In [ ]:
#cell1 环境配置与依赖导入
import sys
import multiprocessing as mp
import time
import numpy as np

# ===================== 可修改配置（已适配10分钟跑完） =====================
class GenConfig:
    # 依赖脚本所在路径：替换为你上传的依赖数据集名称
    dependency_path = "/kaggle/input/chess-ai-scripts/"
    # 输出文件路径（Kaggle仅working目录可写，训练脚本直接读取这个文件）
    out_path = "/kaggle/working/selfplay_data.npz"
    
    # 生成参数（已调优适配10分钟耗时）
    games = 8000          # 总对局数
    workers = 6           # 并行进程数（Kaggle 8核CPU建议设6）
    depth = 2             # 搜索深度
    max_moves = 120       # 单局最大步数
    progress_every = 100  # 每完成多少局打印一次进度

# 添加依赖脚本到系统路径
sys.path.append(GenConfig.dependency_path)
from ai_engine import AIEngine
from chess_game import ChessGame, RED, BLACK
from features import extract_features

# 打印环境信息
print(f"CPU核心数: {mp.cpu_count()}, 启用进程数: {GenConfig.workers}")
print(f"配置参数：对局数={GenConfig.games}, 搜索深度={GenConfig.depth}, 单局上限={GenConfig.max_moves}步")

In [ ]:
#cell2 单局对弈核心函数
def _play_one_game(args: tuple) -> list:
    """子进程内完成一局对弈，返回该局所有带标签的局面数据"""
    _game_idx, depth, max_moves = args
    engine = AIEngine(disable_nn=True)
    game = ChessGame()

    steps = []
    ply = 0
    while not game.game_over and ply < max_moves:
        color = game.current_turn
        feat = extract_features(game)
        board_snapshot = np.array(game.board, dtype=np.int8)
        best_move, move_scores = engine.get_move_with_scores(game, depth=depth)
        if best_move is None:
            break
        moves = list(move_scores.keys())
        scores = [move_scores[m] for m in moves]
        steps.append((feat, board_snapshot, scores, moves, color))
        game.make_move(*best_move)
        ply += 1

    # 生成最终结果标签（红方视角）
    if game.game_over and game.winner is not None:
        result = game.winner
    else:
        result = 0

    labeled = []
    for feat, board_snapshot, scores, moves, color in steps:
        z = result if color == RED else -result
        labeled.append((feat, board_snapshot, scores, moves, z))
    return labeled

In [ ]:
#cell3 并行生成主逻辑
print("=== 开始生成自我对弈数据 ===")
t_start = time.time()

# 生成任务列表
tasks = [(i, GenConfig.depth, GenConfig.max_moves) for i in range(GenConfig.games)]

# 预分配存储列表
all_features = []
all_boards = []
all_move_scores = []
all_moves_order = []
all_z = []

completed = 0
# 多进程并行生成
with mp.Pool(processes=GenConfig.workers) as pool:
    for game_steps in pool.imap_unordered(_play_one_game, tasks):
        # 批量追加数据
        for feat, board_snapshot, scores, moves, z in game_steps:
            all_features.append(feat)
            all_boards.append(board_snapshot)
            all_move_scores.append(scores)
            all_moves_order.append(moves)
            all_z.append(z)
        completed += 1
        
        # 进度打印
        if completed % GenConfig.progress_every == 0 or completed == GenConfig.games:
            elapsed = time.time() - t_start
            rate = completed / elapsed if elapsed > 0 else 0.0
            remaining = (GenConfig.games - completed) / rate if rate > 0 else 0.0
            print(
                f"[{completed}/{GenConfig.games}] 累计局面数={len(all_features)} "
                f"耗时={elapsed:.1f}s 速度={rate:.2f}局/秒 预估剩余={remaining/60:.1f}分钟"
            )

# 转换为numpy数组并保存
print("=== 数据整理中，正在保存文件 ===")
features_arr = np.stack(all_features, axis=0).astype(np.float32)
boards_arr = np.stack(all_boards, axis=0).astype(np.int8)
z_arr = np.array(all_z, dtype=np.float32)
move_scores_arr = np.array(all_move_scores, dtype=object)
moves_order_arr = np.array(all_moves_order, dtype=object)

np.savez(
    GenConfig.out_path,
    features=features_arr,
    boards=boards_arr,
    move_scores=move_scores_arr,
    z_values=z_arr,
    moves_order=moves_order_arr,
)

total_time = time.time() - t_start
print(f"\n✅ 生成完成！")
print(f"总对局数：{GenConfig.games} 局，总局面数：{len(all_features)} 个")
print(f"总耗时：{total_time/60:.1f} 分钟")
print(f"文件已保存至：{GenConfig.out_path}")

In [ ]:
#cell4 数据校验（可选，验证生成的文件是否正常）
# 加载生成的文件检查格式
data = np.load(GenConfig.out_path, allow_pickle=True)
print("=== 数据格式校验 ===")
print(f"features 形状: {data['features'].shape}, 类型: {data['features'].dtype}")
print(f"boards 形状: {data['boards'].shape}, 类型: {data['boards'].dtype}")
print(f"z_values 形状: {data['z_values'].shape}, 类型: {data['z_values'].dtype}")
print(f"move_scores 长度: {len(data['move_scores'])}, 单样本示例长度: {len(data['move_scores'][0])}")
print(f"moves_order 长度: {len(data['moves_order'])}, 单样本示例长度: {len(data['moves_order'][0])}")